# miniLLM 49M — SFT v3

Nouvelle recette : moins de French Alpaca (bruité) et plus de réponses GPT-4, **2 époques seulement** avec un learning rate qui décroît vraiment (1e-4 → 1e-5), weight decay 0.1, dropout 0.1. Part du pré-entraînement terminé (`checkpoints/pretrain/best.pt`) et écrit dans `checkpoints/sft_v3` : aucun ancien run n'est remplacé.

**Utilisation :** Exécution → type d'exécution → GPU (T4), puis **Tout exécuter**. Si la session est coupée, relance.

In [ ]:
# ── miniLLM 49M · SFT v3 (nouvelle recette) ──
# Lance cette cellule (ou « Tout exécuter ») : rien à coller. Ne remplace aucun ancien run (dossier sft_v3).
import os, sys, glob, shutil, json
from google.colab import drive
drive.mount("/content/drive")

cands = (["/content/drive/MyDrive/MiniLLM_v2"] + glob.glob("/content/drive/MyDrive/*/MiniLLM_v2")
         + glob.glob("/content/drive/Shareddrives/*/MiniLLM_v2"))
BASE = next((c for c in cands if os.path.isdir(f"{c}/data")), None)
if BASE is None:
    raise SystemExit("Dossier MiniLLM_v2 introuvable (Drive > Partagés > appui long > Ajouter un raccourci > Mon Drive).")
DATA, PT_OUT = f"{BASE}/data", f"{BASE}/checkpoints/pretrain"
SFT_OUT = f"{BASE}/checkpoints/sft_v3"            # fixe : une relance reprend ce run
CODE_DIR = "/content/minillm_v2"

# Code : branche v2.1.1 (celle de ton notebook : dédoublonnage, sources SFT, évaluation)
!pip -q install tokenizers datasets
if os.path.exists(f"{CODE_DIR}/.git"):
    !git -C {CODE_DIR} fetch -q origin v2.1.1 && git -C {CODE_DIR} checkout -q v2.1.1 && git -C {CODE_DIR} pull -q --ff-only
else:
    !git clone -q -b v2.1.1 https://github.com/bono-p/minillm_v2.git {CODE_DIR}
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)

# Garde-fou : le pré-entraînement doit être terminé
import torch
assert os.path.exists(f"{PT_OUT}/best.pt"), f"{PT_OUT}/best.pt introuvable"
b = torch.load(f"{PT_OUT}/best.pt", map_location="cpu", weights_only=False)
print(f"Base du SFT : pré-entraînement it={b['iter']} | val_loss={b['val_loss']:.4f}")
if int(b["iter"]) < 49_000:
    raise SystemExit("Le pré-entraînement n'est pas terminé (it < 49 000) : laisse-le finir d'abord.")
del b

# 1) Données SFT v3 : moins de French Alpaca (bruité), + réponses GPT-4, persona x8, synthétique dédoublonné
SFT_DATA = f"{DATA}/sft_v3"
if os.path.exists(f"{SFT_DATA}/meta.json"):
    print("✓ Données SFT v3 déjà prêtes :", json.load(open(f"{SFT_DATA}/meta.json"))["sources"])
else:
    !python sft_data.py --tokenizer {DATA}/tokenizer.json --out {SFT_DATA} --alpaca 15000 --oasst 3000 --piaf 2000 --alpaca_gpt4 10000 --evol 0 --sharegpt 0 --persona personnalite.jsonl --persona_repeat 8

LOCAL = "/content/minillm_local"
os.makedirs(LOCAL, exist_ok=True)
if os.path.exists(f"{LOCAL}/sft_v3"):
    shutil.rmtree(f"{LOCAL}/sft_v3")
shutil.copytree(SFT_DATA, f"{LOCAL}/sft_v3")
shutil.copy(f"{DATA}/tokenizer.json", f"{LOCAL}/tokenizer.json")

# 2) Entraînement : 2 époques, LR 1e-4 -> 1e-5 (le LR DÉCROÎT vraiment), weight decay 0.1, dropout d'attention 0.1,
#    évaluation dense au début ; best.pt = meilleur val_loss (pas d'early stopping : reprise sans risque)
!python train.py --mode sft --data_dir {LOCAL}/sft_v3 --out_dir {SFT_OUT} --init_from {PT_OUT}/best.pt \
    --epochs 2 --lr 1e-4 --min_lr 1e-5 --weight_decay 0.1 --dropout 0.1 --warmup_iters 100 --patience 0 \
    --eval_every 250 --eval_every_dense 100 --eval_dense_until 1500 --save_every 250

# 3) Évaluations (décodage glouton) sur le meilleur checkpoint
!python evaluate.py qa      --ckpt {SFT_OUT}/best.pt --sft_dir {LOCAL}/sft_v3 --n 500
!python evaluate.py persona --ckpt {SFT_OUT}/best.pt
!python evaluate.py demo    --ckpt {SFT_OUT}/best.pt
!python evaluate.py openqa  --ckpt {SFT_OUT}/best.pt --n 1000
